# Lesson Brief — Aws Sagemaker
<!-- LESSON_BRIEF_COURSE11 -->

**What you will do:** Work through this example top to bottom. Read each section header before running the next code cell.

**Why it matters:** Students learn how managed cloud platforms change ops, security, and monitoring.

**How to run:** Use Python 3.10+. Run cells in order. If you change data or a model above, use **Kernel → Restart & Run All** before trusting later cells.

**Stuck?** See `Course 11/START_HERE.md` and `Course 11/DOCS/REQUIREMENTS_COURSE_11.md`.

---


# AWS SageMaker Deployment
## AIAT 125 - Model Deployment

## Learning Objectives

- Deploy models on AWS SageMaker
- Use SageMaker endpoints
- Manage model versions
- Scale inference

## Real-World Context

Managed ML platform deployment on AWS.

**Industry Impact**: Used by thousands of companies for production ML.


In [ ]:
%pip install boto3 -q
import boto3
print('✅ boto3 OK:', boto3.__version__)
print('Note: pip install sagemaker is large — run it in AWS CloudShell / SageMaker Studio before real deployment.')
try:
    import sagemaker
    print('✅ sagemaker:', sagemaker.__version__)
except ImportError:
    print('⚠️ sagemaker not installed locally (optional for this concepts notebook).')


## Part 1: SageMaker Concepts


In [ ]:
print('📝 SageMaker Components:')
print('\n1. Training Jobs: Train models')
print('2. Model Registry: Store models')
print('3. Endpoints: Serve predictions')
print('4. Batch Transform: Batch inference')
print('5. Real-time Inference: Low-latency serving')
print('\n✅ SageMaker components understood!')


## Part 2: Deployment Process


In [ ]:
print('📝 SageMaker Deployment Steps:')
print('\n1. Upload model to S3')
print('2. Create model in SageMaker')
print('3. Create endpoint configuration')
print('4. Deploy endpoint')
print('5. Invoke endpoint for predictions')
print('\n✅ Deployment process understood!')
print('\nReal-world: Managed ML platform deployment')


## Real-World Applications

- **Enterprise ML**: Managed ML platform
- **Scalability**: Auto-scaling endpoints
- **Cost Efficiency**: Pay per use
- **Integration**: AWS ecosystem integration

---

**End of Notebook**


## 🌍 Real-World Worked Example — MLflow Experiment Tracking (Production Pattern)

**Industry context:**
- Netflix uses MLflow to track 1000s of A/B test model variants
- Airbnb logs every model training run with parameters, metrics, and artifacts
- Booking.com uses experiment tracking to compare models before deploying to 150M users

We demonstrate **MLflow-style experiment tracking** using Python — the same pattern used in production ML pipelines.


In [ ]:
import time, json, pathlib, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Simulate MLflow experiment tracking (same API pattern)
RUNS_LOG = '/tmp/mlflow_runs.jsonl'
runs = []

def log_run(name, params, metrics, tags=None):
    entry = {
        'run_id': f'run_{len(runs):03d}',
        'name': name,
        'params': params,
        'metrics': metrics,
        'tags': tags or {},
        'timestamp': time.time()
    }
    runs.append(entry)
    with open(RUNS_LOG, 'a') as f:
        f.write(json.dumps(entry) + '\n')
    return entry

# Load dataset (Iris: classic production ML benchmark)
iris = load_iris()
X, y = iris.data, iris.target

# Register experiments (same as mlflow.start_run())
models = {
    'LogisticRegression':   (LogisticRegression(max_iter=200),             {'C': 1.0}),
    'RandomForest_50':      (RandomForestClassifier(n_estimators=50),      {'n_estimators': 50}),
    'RandomForest_100':     (RandomForestClassifier(n_estimators=100),     {'n_estimators': 100}),
    'GradientBoosting':     (GradientBoostingClassifier(n_estimators=50),  {'n_estimators': 50, 'lr': 0.1}),
}

results = []
print("Running experiments...")
for name, (clf, params) in models.items():
    start = time.perf_counter()
    cv_scores = cross_val_score(clf, X, y, cv=5, scoring='accuracy')
    elapsed = time.perf_counter() - start
    metrics = {
        'cv_mean_accuracy': round(cv_scores.mean(), 4),
        'cv_std':           round(cv_scores.std(), 4),
        'training_time_s':  round(elapsed, 3)
    }
    run = log_run(name, params, metrics, tags={'dataset': 'iris', 'framework': 'sklearn'})
    results.append((name, metrics))
    print(f"  [{name:25s}]  acc={metrics['cv_mean_accuracy']:.4f} +/- {metrics['cv_std']:.4f}  | {elapsed:.2f}s")

# Plot: production-style model comparison
names = [r[0].replace('_', ' ') for r in results]
means = [r[1]['cv_mean_accuracy'] for r in results]
stds  = [r[1]['cv_std']           for r in results]
times = [r[1]['training_time_s']  for r in results]
best  = int(np.argmax(means))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
bars = axes[0].bar(names, means, yerr=stds, capsize=5, alpha=0.8)
bars[best].set_color('green'); bars[best].set_label('Best model')
axes[0].set_title("Model Comparison (MLflow-style)"); axes[0].set_ylabel("CV Accuracy"); axes[0].legend()
axes[1].bar(names, times, alpha=0.8)
axes[1].set_title("Training Time"); axes[1].set_ylabel("Seconds")
plt.suptitle("Production ML Experiment Tracking — Same Pattern as Netflix/Airbnb")
plt.tight_layout(); plt.savefig('/tmp/mlflow_comparison.png', dpi=72)

print(f"Best model: {results[best][0]} (accuracy={means[best]:.4f})")
print(f"Experiment log saved to: {RUNS_LOG}")


## 📚 References & Further Reading

**Cloud ML Platforms:**
- [AWS SageMaker](https://docs.aws.amazon.com/sagemaker/)
- [Google Cloud Vertex AI](https://cloud.google.com/vertex-ai/docs)
- [Azure Machine Learning](https://learn.microsoft.com/en-us/azure/machine-learning/)

**MLOps:**
- [MLflow](https://mlflow.org/) — Open-source experiment tracking
- [Weights & Biases](https://wandb.ai/) — Production MLOps platform

**State-of-the-Art:** Netflix, Airbnb, Uber run 1000+ ML models in production using SageMaker/Vertex AI with full MLflow experiment tracking.


## 📝 Summary

You learned **cloud ML deployment** on AWS SageMaker / Azure ML / GCP Vertex AI. Managed endpoints handle auto-scaling, load balancing, and blue-green deployments automatically. Cloud deployment reduces time-to-production from weeks to hours for most ML teams.


## Did you understand? (about 2 minutes)

<!-- STUDENT_SELF_CHECK_COURSE11 -->

Answer **without scrolling** first, then compare with the notebook.

1. **One sentence:** What is the main deployment idea this notebook taught?
2. **Trace one step:** Name one artifact (file, API route, container, or metric) and what role it plays in production.
3. **One question:** What would you ask if you had to deploy this for real users tomorrow?

If any answer is blank, re-run the notebook slowly (one cell → read output → next cell).
